## COMPAS - 2 OBJECTIVES

### Logistic Regression

* **Dataset:** Compass
* **Task:** Classification
* **Target:** Two year recidivism
* **Objectives:** Non-white and White individuals

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import numpy as np
import pandas as pd

from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
from machinemoo import moo
from machinemoo import get_objectives, get_models
from machinemoo.analysis.visualization import plot_pareto, plot_multiple_hypervolumes
from machinemoo.analysis.metrics import compute_hypervolume_progress

from scalarization import LogRegScalarization

### Dataset and Model

In [ ]:
# Setting a seed for reproducibility
seed = 42
#seed = 13

In [ ]:
data = pd.read_csv("dataset/compas_onerace.csv")
data = data.drop("Unnamed: 0", axis=1)

fair_feature = "not_white"
pred_feature = "Two_yr_Recidivism"

categories_fair_class = []

for index, row in data.iterrows():
    if row[pred_feature] == -1:
        categories_fair_class.append(row[fair_feature])
    else:
        categories_fair_class.append(row[fair_feature]+2)

X = data.drop([pred_feature], axis=1)
# y = data[pred_feature]
y = (data[pred_feature] + 1)//2

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = int(data.shape[0]*0.5),
                                                    stratify = categories_fair_class,
                                                    random_state = seed)


In [ ]:
#tol = 10**-3
#max_iter = 100

tol = 10**-8
max_iter = 10

## MOO Modeling and Training

* MOO methods:
    * MOLA
    * MONISE
    * Random Weights

In [ ]:
# Setting optimization parameter to use in all methods in this experiment
opt_params = {
    'node_time_limit': 2,
    'target_size': 50,
    'target_gap': 0,
    'node_gap': 0.05,
    'norm': False
}
num_objs = 2
results = {}

In [ ]:
# Logistic Regression with equal weight to use as baseline model
equal = LogRegScalarization(num_objs=2,
                            X=X_train, 
                            y=y_train, 
                            fair_feat=fair_feature,
                            #tol=tol, max_iter=max_iter
                            )
equal.optimize(np.array([0.5, 0.5]))

### MOLA

In [ ]:
method = 'mola'
w_scalar = LogRegScalarization(num_objs=2, 
                               X=X_train, 
                               y=y_train,
                               fair_feat=fair_feature,
                               tol=tol,
                               max_iter=max_iter, 
                               gradient=False)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
#hypervolume_values = compute_hypervolume_progress(objs)
hypervolume_values = moopt.get_hypervolumes()
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": get_models(moopt),
                "hypervolume": hypervolume_values
            }

In [82]:
import plotly.graph_objects as go

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go



In [119]:
title="Conflict between losses for white and non-white person in training"
plot_pareto_plotly_2d(methods={'mola': objs, 'mola2': objs, 'mola3': objs},
            labels=("White", "Non-White"), 
            # title=title,
            point=equal.objs,
            # color="#FF5C8D",
            fontsize=20,
            figsize=(15,7),
            save_path= None #'images/2_objs_mola.pdf'
            )

In [120]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_pareto_plotly_3d(
        methods,
        point     = None,
        save_path = None,
        labels    = None,
        color     = ['#FF5C8D', '#4BD6A0', '#9D5CFF'],
        point_label = "Baseline Model",
        title       = "Pareto Frontier",
        fontsize    = 20,
        figsize     = (10, 7),
) -> None:

    num_plots = len(methods)
    
    layout_configs = {
        1: (1, 1),
        2: (1, 2),
        3: (1, 3),
        4: (2, 2),
        5: (2, 3),
        6: (2, 3),
        7: (3, 3),
        8: (3, 3),
        9: (3, 3)
    }

    rows, cols = layout_configs[num_plots]
    positions  = [(i,j) for i in range(rows) for j in range(cols)]

    first_key      = next(iter(methods))
    num_objectives = methods[first_key].shape[1]
    default_labels = [f"Objective {i+1}" for i in range(num_objectives)]
    labels         = labels if labels else default_labels
    
    fig = make_subplots(rows=rows, 
                        cols=cols, 
                        subplot_titles=list(methods.keys()),
                        specs=[[{'type': 'scene'} for _ in range(cols)] for _ in range(rows)])

    for i, ((row, col), (key, values)) in enumerate(zip(positions, methods.items())):
        scene_id = f'scene{(row * cols + col + 1)}'
        fig.add_trace(
            go.Scatter3d(
                x=values[:, 0],
                y=values[:, 1],
                z=values[:, 2],
                mode='markers',
                marker=dict(size=12, 
                            color=color[i % len(color)], 
                            line=dict(color='black', width=1))
            ),
            row=row + 1,
            col=col + 1
        )
    
        if point is not None:
            fig.add_trace(
                go.Scatter3d(
                    x=[point[0]],
                    y=[point[1]],
                    z=[point[2]],
                    mode='markers+text',
                    name=point_label,
                    marker=dict(size=15, color='black', symbol='star'),
                    text=[point_label],
                    textposition='top right',
                ),
                row=row+1,
                col=col+1,
            )

        fig.update_layout({
            scene_id: dict(
                xaxis_title=labels[0],
                yaxis_title=labels[1],
                zaxis_title=labels[2],
                xaxis=dict(showgrid=True, gridcolor='lightgray'),
                yaxis=dict(showgrid=True, gridcolor='lightgray'),
                zaxis=dict(showgrid=True, gridcolor='lightgray'),
            )
        })

    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=fontsize, color='black')

    fig.update_layout(
        height=int(figsize[1] * 100),
        width=int(figsize[0] * 100),
        title=dict(text=title, x=0.5, xanchor='center', font=dict(size=fontsize)),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=fontsize, color='black'),
        margin=dict(l=60, r=40, t=80, b=60),
        showlegend=False
    )
    
    fig.show()
    
    if save_path is not None:
        fig.write_html(save_path)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_pareto_plotly_4d(
        methods,
        point     = None,
        save_path = None,
        labels    = None,
        color     = ['#FF5C8D', '#4BD6A0', '#9D5CFF'],
        point_label = "Baseline Model",
        title       = "Pareto Frontier",
        fontsize    = 20,
        figsize     = (10, 7),
) -> None:

    num_plots = len(methods)

    layout_configs = {
        1: (1, 1),
        2: (1, 2),
        3: (1, 3),
        4: (2, 2),
        5: (2, 3),
        6: (2, 3),
        7: (3, 3),
        8: (3, 3),
        9: (3, 3)
    }

    rows, cols = layout_configs[num_plots]
    positions  = [(i, j) for i in range(rows) for j in range(cols)]

    first_key      = next(iter(methods))
    num_objectives = methods[first_key].shape[1]

    default_labels = [f"Objective {i+1}" for i in range(num_objectives)]
    labels         = labels if labels else default_labels

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=list(methods.keys()),
        specs=[[{'type': 'domain'} for _ in range(cols)] for _ in range(rows)]
    )

    for i, ((row, col), (key, values)) in enumerate(zip(positions, methods.items())):
        dimensions = [
            dict(label=labels[d], values=values[:, d])
            for d in range(values.shape[1])
        ]

        fig.add_trace(
            go.Parcoords(
                line=dict(color=color[i % len(color)]),
                dimensions=dimensions
            ),
            row=row + 1,
            col=col + 1
        )

    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=fontsize, color='black')

    fig.update_layout(
        height=int(figsize[1] * 100),
        width=int(figsize[0] * 100),
        title=dict(text=title, x=0.5, xanchor='center', font=dict(size=fontsize)),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=fontsize, color='black'),
        margin=dict(l=60, r=40, t=80, b=60),
        showlegend=False
    )

    fig.show()

    if save_path is not None:
        fig.write_html(save_path)


In [130]:
np.concat((objs, objs), axis=1).shape

(48, 4)

In [135]:
title="Conflict between losses for white and non-white person in training"
plot_pareto_plotly_4d(methods={'mola': np.concat((objs, objs), axis=1), 'mola2':  np.concat((objs, objs), axis=1)},
            labels=("White", "Non-White", 'Super White', 'Godly White'), 
            # title=title,
            point=equal.objs,
            # color="#FF5C8D",
            fontsize=20,
            figsize=(15,7),
            save_path= None #'images/2_objs_mola.pdf'
            )

(48, 4)
(48, 4)


In [ ]:
fig = go.Figure()

fig.add_trace(go.Parcoords(
#    line=dict(
#        color=objs[:, 0],
#        colorscale='RdPu',
#        showscale=True,
#        cmin=objs[:, 0].min(),
#        cmax=objs[:, 0].max()
#    ),
    line=dict(color='#FF5C8D'),
    dimensions=[
        dict(label='Dimensão 1', values=objs[:, 0]),
        dict(label='Dimensão 2', values=objs[:, 1]),
        dict(label='Dimensão 3 (igual à 2)', values=objs[:, 1]),
        dict(label='Dimensão 4 (igual à 1)', values=objs[:, 0])
    ]
))

fig.update_layout(
    title='Gráfico de Coordenadas Paralelas',
    title_x=.5,
    title_font=dict(size=22, color='#333', family='Arial'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=14, color='black'),
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.show()


In [ ]:
np.arange(min(objs[:, 0]), max(objs[:, 0]), 0.0001).round(4)

In [ ]:
min(objs[:, 0])

In [ ]:
max(objs[:, 0])

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Parcoords(
    line=dict(color='#FF5C8D'),
    dimensions=[
        dict(label='Dim 1', values=objs[:, 0]),
        dict(label='Dim 2', values=objs[:, 1]),
        dict(label='Dim 3', values=objs[:, 1]),
        dict(label='Dim 4', values=objs[:, 0])
    ]
))

fig.update_layout(
    title='Gráfico de Coordenadas Paralelas',
    title_x=0.5,
    title_font=dict(size=26, color='#333', family='Arial'),
    font=dict(family='Arial', size=18, color='black'),  # Aplica ao texto geral
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=80, r=80, t=100, b=60)
)

fig.show()


In [ ]:
title="Conflict between losses for white and non-white person in training"
plot_pareto(methods={'mola': objs},
            labels=("White", "Non-White"), 
            # title=title,
            point=equal.objs,
            # color="#FF5C8D",
            fontsize=23,
            figsize=(10,7),
            save_path='images/2_objs_mola.pdf'
            )

In [ ]:
# Test with gradient
method = 'mola'
w_scalar = LogRegScalarization(num_objs=2, 
                               X=X_train, 
                               y=y_train,
                               fair_feat=fair_feature,
                               tol=tol,
                               max_iter=max_iter, 
                               gradient=False)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
hypervolume_values = moopt.get_hypervolumes()
results[f"{method}_grad"] = {
                "moopt": moopt,
                "objectives": objs,
                "models": get_models(moopt),
                "hypervolume": hypervolume_values
            }

In [ ]:
title="Conflict between losses for white and non-white person in training"
plot_pareto(methods={'mola_grad': objs},
            labels=("White", "Non-White"), 
            # title=title,
            point=equal.objs,
            # color="#FF5C8D",
            fontsize=23,
            figsize=(10,7),
            save_path='images/2_objs_mola_grad.pdf'
            )

### MONISE

In [ ]:
scalarization = None
moopt = None
objs = None

In [ ]:
method = 'monise'
w_scalar = LogRegScalarization(num_objs=2, 
                               X=X_train, 
                               y=y_train,
                               fair_feat=fair_feature,
                               tol=tol,
                               max_iter=max_iter, 
                               gradient=False)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
hypervolume_values = compute_hypervolume_progress(objs)
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": get_models(moopt),
                "hypervolume": hypervolume_values
            }

In [ ]:
scalarization = None
moopt = None
objs = None

### Ramdon Weighted Sum

In [ ]:
method = 'random_weight'
w_scalar = LogRegScalarization(num_objs=2, 
                               X=X_train, 
                               y=y_train,
                               fair_feat=fair_feature,
                               tol=tol,
                               max_iter=max_iter, 
                               gradient=False)
moopt = moo(w_scalar).mo_optimization(method, **opt_params)
objs = get_objectives(moopt)
hypervolume_values = compute_hypervolume_progress(objs)
results[method] = {
                "moopt": moopt,
                "objectives": objs,
                "models": get_models(moopt),
                "hypervolume": hypervolume_values
            }

## A posteriori Decision Making

* Comparison between MOO methods
* Ensemble and Evaluation

In [ ]:
from machinemoo.tests.experiments.run import run_experiments

In [ ]:
pareto_dict = {
        method: res["objectives"]
        for method, res in results.items()
        if "objectives" in res
    }

hypervolumes_dict = {
        method: res["hypervolume"]
        for method, res in results.items()
        if "hypervolume" in res
    }

colors_pareto = ['#FF5C8D', '#FF5C8D', '#FF5C8D']
colors_hv = ['#FF5C8D', '#FF5C8D', '#4BD6A0', '#9D5CFF']
title = ""
axes_label = ["White", "Non-White"]

plot_pareto(methods=pareto_dict,
            labels=axes_label, 
            # title=title,
            point=equal.objs,
            # color="#FF5C8D",
            fontsize=23,
            figsize=(10,7),
            save_path='images/2_objs_methods.pdf'
            )

plot_multiple_hypervolumes(methods_hv=hypervolumes_dict)

In [ ]:
from sklearn.metrics import accuracy_score
from machinemoo import run_ensemble

In [ ]:
lambd = math.exp(-100)
clf = LogisticRegression(solver = 'lbfgs',
                           class_weight = None,
                           penalty = 'l2',
                           max_iter = max_iter,
                           tol = tol, 
                           C = 1/lambd,
                           warm_start = True)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
#predprob = clf.predict_proba(X_test)
accuracy = accuracy_score(y_test, pred)

In [ ]:
#Logistic Regression Single Objctive
print(f'Logistic Regression Accuracy: {accuracy:.4f}')

In [ ]:
ensembles = {}
for method in results.keys():
    ensembles[method] = {
        'acc_voting'   : run_ensemble(optimizer=None, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, models=results[method]['models'], ensemble_type='voting'),
        'acc_baggin'   : run_ensemble(optimizer=None, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, models=results[method]['models'], ensemble_type='baggin'),
        'acc_adaboost' : run_ensemble(optimizer=None, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, models=results[method]['models'], ensemble_type='adaboost')
    }

In [ ]:
for method, ensemble in ensembles.items():
    print(f'\nMethod {method.upper()}')
    print(f'  Voting Accuracy:   {ensemble["acc_voting"]:.4f}')
    print(f'  Bagging Accuracy:  {ensemble["acc_baggin"]:.4f}')
    print(f'  Adaboost Accuracy: {ensemble["acc_adaboost"]:.4f}')